# Stage 3: FinBERT Scoring and Daily News Features

Load raw GDELT article metadata, score titles with FinBERT, and save daily stock-level sentiment features.

In [ ]:
import subprocess
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = ROOT / 'data' / 'intermediate'

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'torch', 'sentencepiece'])

articles = pd.read_parquet(DATA_DIR / 'gdelt_articles_raw.parquet')
if articles.empty:
    raise ValueError('No articles found. Run 02_gdelt_collection.ipynb first.')

articles['date'] = pd.to_datetime(articles['date'])
print('Articles loaded:', len(articles))

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

MODEL_NAME = 'yiyanghkust/finbert-tone'
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model.eval()

def finbert_score(texts, batch_size=64):
    label_weight = {0: 0.0, 1: 1.0, 2: -1.0}
    out = np.empty(len(texts), dtype=np.float32)
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors='pt')
        with torch.no_grad():
            logits = model(**enc).logits
        probs = torch.softmax(logits, dim=1).numpy()
        for j, p in enumerate(probs):
            out[i + j] = sum(label_weight[k] * p[k] for k in label_weight)
    return out

In [ ]:
titles = articles['title'].fillna('').astype(str)
unique_titles = titles.drop_duplicates().tolist()

score_map = {}
batch = 64
for i in tqdm(range(0, len(unique_titles), batch), desc='FinBERT scoring'):
    chunk = unique_titles[i:i + batch]
    scores = finbert_score(chunk, batch_size=batch)
    for t, s in zip(chunk, scores):
        score_map[t] = float(s)

articles['finbert_score'] = titles.map(score_map).astype(float)
articles.to_parquet(DATA_DIR / 'gdelt_articles_scored.parquet')
print('Saved gdelt_articles_scored.parquet')

In [ ]:
daily = (
    articles.groupby(['ticker', 'date'])['finbert_score']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'finbert_daily', 'count': 'news_count'})
)
daily.index.set_names(['ticker', 'date'], inplace=True)

finbert_wide = daily['finbert_daily'].unstack('ticker')
count_wide = daily['news_count'].unstack('ticker').fillna(0)

daily.to_parquet(DATA_DIR / 'news_daily_long.parquet')
finbert_wide.to_parquet(DATA_DIR / 'finbert_daily_wide.parquet')
count_wide.to_parquet(DATA_DIR / 'news_count_wide.parquet')

print('Saved news_daily_long.parquet, finbert_daily_wide.parquet, news_count_wide.parquet')
print('Daily long rows:', len(daily))